### LOAD DATA ###

In [3]:
import torch
import numpy as np
import random
import pandas as pd
import os

from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from torch.optim import AdamW
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from tqdm import tqdm

In [4]:
def set_seed_ultimate(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # Buat multi-GPU
    
    # INI KUNCINYA BIAR GPU GAK RANDOM
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Kunci hashing Python
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # Fix algoritma PyTorch biar deterministic
    # os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    # torch.use_deterministic_algorithms(True) # Opsional, kadang bikin lemot

set_seed_ultimate(42)

In [5]:
train_df = pd.read_csv("../data_labelling/train_labeled.csv")
val_df   = pd.read_csv("../data_labelling/val_labeled.csv")
test_df  = pd.read_csv("../data_labelling/test_labeled.csv")

train_df["label"] = train_df["label"].astype(int)
val_df["label"]   = val_df["label"].astype(int)
test_df["label"]  = test_df["label"].astype(int)

print("Distribusi Train:")
print(train_df["label"].value_counts())

Distribusi Train:
label
0    597
2    289
1    214
Name: count, dtype: int64


In [6]:
MODEL_NAME = "indobenchmark/indobert-large-p1"
MAX_LEN = 128
BATCH_SIZE = 8   # kecil karena large
EPOCHS = 5
LR = 2e-5
NUM_LABELS = 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [8]:
class SentimentDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["cleaned_text"].values
        self.labels = dataframe["label"].values
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        
        return item

In [9]:
train_dataset = SentimentDataset(train_df)
val_dataset   = SentimentDataset(val_df)
test_dataset  = SentimentDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-large-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((1

In [11]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=train_df["label"].unique(),
    y=train_df["label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

print("Class Weights:", class_weights)

Class Weights: tensor([0.6142, 1.2687, 1.7134], device='cuda:0')


In [12]:
optimizer = AdamW(model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

In [13]:
ACCUMULATION_STEPS = 4
best_val_loss = float('inf') # Variabel buat nyimpen loss terbaik

# Penyesuaian total_steps buat scheduler karena ada akumulasi
total_steps = (len(train_loader) // ACCUMULATION_STEPS) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

for epoch in range(EPOCHS):
    
    # ===== TRAIN =====
    model.train()
    total_train_loss = 0
    
    loop = tqdm(train_loader, leave=True)
    optimizer.zero_grad() # Pindah ke luar loop batch
    
    for step, batch in enumerate(loop):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        logits = outputs.logits
        loss = criterion(logits, labels)
        
        # Dibagi accumulation steps biar gradiennya gak meledak pas diakumulasi
        loss = loss / ACCUMULATION_STEPS
        loss.backward()
        
        total_train_loss += loss.item() * ACCUMULATION_STEPS
        
        # Update bobot HANYA JIKA udah mencapai accumulation steps
        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad() # Reset gradien setelah update
        
        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item() * ACCUMULATION_STEPS)
    
    avg_train_loss = total_train_loss / len(train_loader)
    
    # ===== VALIDATION =====
    model.eval()
    total_val_loss = 0
    val_preds = []
    val_labels = []
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            logits = outputs.logits
            loss = criterion(logits, labels)
            
            total_val_loss += loss.item()
            
            preds = torch.argmax(logits, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
    
    avg_val_loss = total_val_loss / len(val_loader)
    
    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss  : {avg_val_loss:.4f}")
    print(classification_report(val_labels, val_preds))

    # ===== SAVE BEST MODEL (EARLY STOPPING LOGIC) =====
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_indobert_model.pt')
        print(f"🔥 Val Loss turun! Menyimpan model terbaik di Epoch {epoch+1}...")

print("Training Selesai!")

# ===== TESTING PAKAI MODEL TERBAIK =====
from sklearn.metrics import classification_report, confusion_matrix

print("\nLoad model terbaik untuk Testing...")
# Load bobot (weight) dari epoch yang punya Val Loss terendah
model.load_state_dict(torch.load('best_indobert_model.pt'))
model.eval()

test_preds = []
test_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)
        
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

print("=== TEST RESULT (BEST MODEL) ===")
print(classification_report(test_labels, test_preds))
print("Confusion Matrix:")
print(confusion_matrix(test_labels, test_preds))

Epoch 1: 100%|██████████| 138/138 [14:56<00:00,  6.50s/it, loss=0.947]



Epoch 1
Train Loss: 1.0677
Val Loss  : 0.9568
              precision    recall  f1-score   support

           0       0.91      0.41      0.56       152
           1       0.45      0.25      0.32        40
           2       0.31      0.86      0.46        58

    accuracy                           0.49       250
   macro avg       0.56      0.51      0.45       250
weighted avg       0.70      0.49      0.50       250

🔥 Val Loss turun! Menyimpan model terbaik di Epoch 1...


Epoch 2: 100%|██████████| 138/138 [15:09<00:00,  6.59s/it, loss=0.28] 



Epoch 2
Train Loss: 0.6728
Val Loss  : 0.7680
              precision    recall  f1-score   support

           0       0.87      0.80      0.83       152
           1       0.48      0.60      0.53        40
           2       0.62      0.66      0.64        58

    accuracy                           0.73       250
   macro avg       0.66      0.68      0.67       250
weighted avg       0.75      0.73      0.74       250

🔥 Val Loss turun! Menyimpan model terbaik di Epoch 2...


Epoch 3: 100%|██████████| 138/138 [15:03<00:00,  6.55s/it, loss=1.51] 



Epoch 3
Train Loss: 0.4043
Val Loss  : 0.7626
              precision    recall  f1-score   support

           0       0.84      0.75      0.79       152
           1       0.49      0.57      0.53        40
           2       0.57      0.67      0.62        58

    accuracy                           0.70       250
   macro avg       0.64      0.67      0.65       250
weighted avg       0.72      0.70      0.71       250

🔥 Val Loss turun! Menyimpan model terbaik di Epoch 3...


Epoch 4: 100%|██████████| 138/138 [15:06<00:00,  6.57s/it, loss=0.0639]



Epoch 4
Train Loss: 0.2124
Val Loss  : 0.8137
              precision    recall  f1-score   support

           0       0.87      0.71      0.78       152
           1       0.50      0.62      0.56        40
           2       0.55      0.72      0.63        58

    accuracy                           0.70       250
   macro avg       0.64      0.69      0.66       250
weighted avg       0.74      0.70      0.71       250



Epoch 5: 100%|██████████| 138/138 [15:10<00:00,  6.60s/it, loss=0.123] 



Epoch 5
Train Loss: 0.1278
Val Loss  : 0.8630
              precision    recall  f1-score   support

           0       0.85      0.77      0.81       152
           1       0.52      0.65      0.58        40
           2       0.60      0.66      0.63        58

    accuracy                           0.72       250
   macro avg       0.66      0.69      0.67       250
weighted avg       0.74      0.72      0.73       250

Training Selesai!

Load model terbaik untuk Testing...
=== TEST RESULT (BEST MODEL) ===
              precision    recall  f1-score   support

           0       0.83      0.77      0.80       130
           1       0.72      0.57      0.64        60
           2       0.63      0.87      0.73        60

    accuracy                           0.74       250
   macro avg       0.73      0.73      0.72       250
weighted avg       0.76      0.74      0.74       250

Confusion Matrix:
[[100  10  20]
 [ 15  34  11]
 [  5   3  52]]


In [14]:
from sklearn.metrics import classification_report, confusion_matrix
model.eval()
test_preds = []
test_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)
        
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

print("=== TEST RESULT ===")
print(classification_report(test_labels, test_preds))
print("Confusion Matrix:")
print(confusion_matrix(test_labels, test_preds))

=== TEST RESULT ===
              precision    recall  f1-score   support

           0       0.83      0.77      0.80       130
           1       0.72      0.57      0.64        60
           2       0.63      0.87      0.73        60

    accuracy                           0.74       250
   macro avg       0.73      0.73      0.72       250
weighted avg       0.76      0.74      0.74       250

Confusion Matrix:
[[100  10  20]
 [ 15  34  11]
 [  5   3  52]]


In [15]:
model.save_pretrained("./model1_manual_final")
tokenizer.save_pretrained("./model1_manual_final")

('./model1_manual_final\\tokenizer_config.json',
 './model1_manual_final\\special_tokens_map.json',
 './model1_manual_final\\vocab.txt',
 './model1_manual_final\\added_tokens.json',
 './model1_manual_final\\tokenizer.json')

### CIHUY ###

In [16]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

load_path = "./model1_manual_final"

tokenizer = AutoTokenizer.from_pretrained(load_path)
model = AutoModelForSequenceClassification.from_pretrained(load_path)

model.to(device)
model.eval()

print("Model loaded successfully!")

Model loaded successfully!
